# Building Reasoning Models using GRPO

🌟 This Notebook is adapted from the Hugging Face [Open R1 / GRPO tutorial](https://huggingface.co/learn/llm-course/chapter12/1). By completing the training steps in this notebook, you will be able to train a LLM that is able to "think" and reason on complex problems solely based on reinforcement post-training with verified rewards (RLVR).


## RLHF Pipeline

Recall that creating a ChatGPT at home involves 3 steps:

1. pre-training a large language model (LLM) to predict the next token on internet-scale data, on clusters of thousands of GPUs. One calls the result a "base model"
2. supervised fine-tuning (SFT) to turn the base model into a useful assistant
3. human preference fine-tuning which increases the assistant's friendliness, helpfulness and safety.

In this notebook, we're still going to **illustrate step 3**. Different from DPO, where we fine-tuned a supervised fine-tuned (SFT) model on **human preferences**, we want to further **incentize the reasoning abilities** of the model so that the model is able to solve complex problems,
leveraging a method called [GRPO](https://arxiv.org/abs/2402.03300).

### Reinforcement Post-training with Verified Rewards and GRPO

Recall that with DPO (human preference fine-tuning), we turn the supervised fine-tuned chatbot into a chatbot that behaves in a way we want, like "safe", "friendly", "harmless", "inclusive", or whatever properties we would like our chatbot to have. However, those human preference objectives might not necessarily lead to a model that **reason better**. Hence, reseachers leveraged **reinforcement post-training with verified rewards** to make the chatbot think deeper on complex problems.

To do this, we first need to define what is the **good** response. As many reasoning tasks (such as math, code, logic and games) often have automated verifiable rewards, we can easily evaluate the **correctness** of the response. We can also design other reward functions such as a format reward which check whether the reasoning content is placed within the `<think>...</think>` XML tags and the answer is placed within the `<answer>...</answer>` tags, or a length reward function that prefer shorter response to mitigate the issue of [over-thinking](https://arxiv.org/abs/2412.21187). The final overall reward score can be a (weighted) mixture of different reward functions.


We will then fine-tune the SFT model to **make it learn to generate responses that achieve high rewards**. The method we use is GRPO, which generate a group of responses for a given question during training and compute their corresponding rewards.

### Algorithm: GRPO

**Input**

- `initial_policy`: starting model to be trained  
- `reward_function`: function that evaluates outputs  
- `training_prompts`: set of training examples  
- `group_size`: number of outputs per prompt (typically 4–16)  

---

**Procedure**

1. For each training iteration:
   1. Set `reference_policy = initial_policy`  (snapshot current policy)

   2. For each prompt `x` in the current batch:
      1. **Sample outputs**
         - Generate `group_size` different outputs using `initial_policy`.
      2. **Compute rewards**
         - For each output `y_i`, compute  
           `r_i = reward_function(x, y_i)`
      3. **Normalize rewards within the group**
         - Compute mean and standard deviation:  
           `mu = mean(r_1, ..., r_group_size)`  
           `sigma = std(r_1, ..., r_group_size)`  
         - For each `i`, define the normalized advantage:  
           `A_i = (r_i - mu) / sigma`
         - Note that the standard deviation term is used by DeepSeek in their DeepSeek-Math paper. Later works such as [Dr.GRPO](https://arxiv.org/abs/2503.20783) argue that removing this term can help training stability.
      4. **Compute policy update objective**
         - For each sample, compute the probability ratio:  
           `prob_ratio_i = p_current(y_i | x) / p_reference(y_i | x)`
         - Use the clipped surrogate objective:  
           `L_i = min(prob_ratio_i * A_i, clip(prob_ratio_i, 1 - epsilon, 1 + epsilon) * A_i)`
         - Add KL regularization term:  
           `L_i = L_i - kl_weight * KL(initial_policy || reference_policy)`
      5. **Update policy**
         - Update model parameters by maximizing the sum of `L_i` over the batch.

---

**Output**

- `optimized_policy`: the trained policy model


A nice collection of openly available reasoning datasets collected by the Hugging Face team can be found [here](https://huggingface.co/collections/open-r1/reasoning-datasets), which mainly contains math and code problems. For other open-domain questions, [Reasoning Gym](https://github.com/open-thought/reasoning-gym) is another good resource.

We will perform RL post-training starting from the [`Qwen2 0.5B Instruct`](https://huggingface.co/Qwen/Qwen2-0.5B-Instruct) model on the [`DeepMath-103K`](https://huggingface.co/datasets/trl-lib/DeepMath-103K) dataset to incentize and enlarge the mathematical reasoning ability of the model.

Notes:

* the entire notebook is based on and can be seen as an annotated version of the [Open R1](https://github.com/huggingface/open-r1) developed by Hugging Face. Huge kudos to the team for creating this! If you want to conduct researsch in this area, Open R1 is one of the best codebases to start with. Other codebase such as [VeRL](https://github.com/volcengine/verl) provides more recipes on different models and different RL algorithms that can start training with only several lines of coding, but are also more complex and difficult to have fine-grained control.

## Required hardware

The notebook is designed to be run on any NVIDIA GPU which has the [Ampere architecture](https://en.wikipedia.org/wiki/Ampere_(microarchitecture)) or later with at least 24GB of RAM. This includes:

* NVIDIA RTX 3090, 4090
* NVIDIA A100, H100, H200

and so on. Try the A100 GPU that can be acquired via Colab Pro.

The reason for an Ampere requirement is because we're going to use the [bfloat16 (bf16) format](https://en.wikipedia.org/wiki/Bfloat16_floating-point_format), which is not supported on older architectures like Turing.

But: a few tweaks can be made to train the model in float16 (fp16), which is supported by older GPUs like:

* NVIDIA RTX 2080
* NVIDIA Tesla T4
* NVIDIA V100.

Comments are added regarding where to swap bf16 with fp16.

## Set-up environment

Let's start by installing all the 🤗 goodies we need to do reinforcement post-training. We're going to use

* Transformers for the LLM which we're going to fine-tune
* Datasets for loading a human preference dataset from the 🤗 hub, and preparing it for the model
* BitsandBytes and PEFT for fine-tuning the model on consumer hardware, leveraging [Q-LoRa](https://huggingface.co/blog/4bit-transformers-bitsandbytes), a technique which drastically reduces the compute requirements for fine-tuning
* TRL, a [library](https://huggingface.co/docs/trl/index) which includes useful Trainer classes for LLM fine-tuning, including GRPO.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers[torch] datasets

In [ ]:
!pip install -q bitsandbytes trl peft

We also install [Flash Attention](https://github.com/Dao-AILab/flash-attention), which speeds up the attention computations of the model. However, running the `!pip install flash-attn --no-build-isolation` on Colab takes too long time (more than 1 hours) to recompile the flash attention package. So we can skip it if you don't want to wait here.

In [ ]:
!pip install setuptools wheel packaging ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.8 MB/s eta 0:00:00


In [ ]:
!pip install flash-attn --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 146.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user
^C


In [ ]:
!pip install math-verify

## Load dataset

As for the dataset, we need one containg questions and correct answers. Here we will load the [trl-lib/DeepMath-103K](https://huggingface.co/datasets/trl-lib/DeepMath-103K) dataset. This dataset is a post-processed version of the original [DeepMath-103K](https://huggingface.co/datasets/zwhe99/DeepMath-103K) dataset.

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("trl-lib/DeepMath-103K")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 97870
    })
    test: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 5152
    })
})

The dataset contains various splits, each with a certain number of rows with fields including `prompt` and `solution`.

In [ ]:
from datasets import DatasetDict

# We filter out examples that has mathematical verifiable solutions
def has_latex_solution(example):
    # defensive: make sure it's a string
    sol = example["solution"]
    return isinstance(sol, str) and sol.startswith("$")

filtered = raw_datasets.filter(has_latex_solution)

# For debugging, we only select the first 100 examples.
indices = range(0,100)

dataset_dict = {"train": filtered["train"].select(indices),
                "test": filtered["test"].select(indices)}

# dataset_dict = {"train": filtered["train"],
#                 "test": filtered["test"]}

raw_datasets = DatasetDict(dataset_dict)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 100
    })
    test: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 100
    })
})

Let's check one example. The important thing is that each training example should contain the following 2 things:

* a prompt (question)
* a solution (ground-truth correct answer)

In [ ]:
from pprint import pprint

example = raw_datasets["train"][0]
print(example.keys())

dict_keys(['prompt', 'solution'])


Let's see what the question and solution was in this case:

In [ ]:
example["prompt"]

[{'content': 'Find the distance from the origin to the curve defined by the equations \\( \\frac{x^2}{4} + y^2 + \\frac{z^2}{4} = 1 \\) and \\( x + y + z = 1 \\) in \\( \\mathbb{R}^3 \\).',
  'role': 'user'}]

In [ ]:
example["solution"]

'$1$'

In [ ]:
example = raw_datasets["train"][1]
pprint(example['prompt'])

[{'content': 'Compute the limit: $$\\lim_{x\\rightarrow 0}\\frac{3x^2-3x\\sin '
             'x}{x^2+x\\cos\\frac{1}{x}}$$',
  'role': 'user'}]


In [ ]:
pprint(example['solution'])

'$0$'


## Load tokenizer

Next, we instantiate the tokenizer, which is required to prepare the texts for the model. The model doesn't directly take strings as input, but rather `input_ids`, which represent integer indices in the vocabulary of a Transformer model. Refer to this [YouTube video](https://www.youtube.com/watch?v=IGu7ivuy1Ag&ab_channel=NielsRogge) if you want to know more about it.

We also set some attributes which the tokenizer of a base model typically doesn't have set, such as:

- the padding token ID. During pre-training, one doesn't need to pad since one just creates blocks of text to predict the next token, but during fine-tuning, we will need to pad the (prompt, response) pairs in order to create batches of equal length. This is also needed in GRPO: we will (separately) collect lots of rollouts during online inference, but when we perform policy gradient, we are still doing batch trainig, so we still need the pad token.
- the truncation side: when sequences are too long, they need to be truncated to fit the same length. Here we make sure to truncate from the **right**, to make sure we don't lose the content of the question. This also "implicitly" penalize those responses that are too long.
- the model max length: this is required in order to pad/truncate sequences which are too long for the model. Here we decide to train on at most 2048 tokens.
- the chat template. A [chat template](https://huggingface.co/blog/chat-templates) determines how each list of messages is turned into a tokenizable string, by adding special strings in between such as `<|user|>` to indicate a user message and `<|assistant|>` to indicate the chatbot's response. Different from the chat template we used for DPO preference alignment, here we define the default chat template, used by Qwen-series models. See also the [docs](https://huggingface.co/docs/transformers/main/en/chat_templating).

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2-0.5B-Instruct"    # This is the model id of the SFT model, we will use the same tokenizer here for consistency.

tokenizer = AutoTokenizer.from_pretrained(model_id)   # Load the tokenizer

In [ ]:
tokenizer.pad_token

'<|endoftext|>'

In [ ]:
tokenizer.eos_token

'<|im_end|>'

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2-0.5B-Instruct"    # This is the model id of the SFT model, we will use the same tokenizer here for consistency.

tokenizer = AutoTokenizer.from_pretrained(model_id)   # Load the tokenizer

# As noted above, since we are perform batch training, we need the <pad> tokens
# to help the short completions match the length of the long completions.
# If no specific pad token is being used, by default we use the <eos> token.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Truncate from right to ensure we don't lose content of the question
tokenizer.truncation_side = "right"
tokenizer.padding_side='left'

# Set reasonable default for models without max length
if tokenizer.model_max_length > 100_000:
    tokenizer.model_max_length = 2048

# We set the default chat template to be the template of Qwen model
DEFAULT_CHAT_TEMPLATE = "{%- if tools %}\n    {{- '<|im_start|>system\\n' }}\n    {%- if messages[0]['role'] == 'system' %}\n        {{- messages[0]['content'] }}\n    {%- else %}\n        {{- 'Please reason step by step, and put your final answer within \\\\boxed{}.' }}\n    {%- endif %}\n    {{- \"\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>\" }}\n    {%- for tool in tools %}\n        {{- \"\\n\" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- \"\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\\"name\\\": <function-name>, \\\"arguments\\\": <args-json-object>}\\n</tool_call><|im_end|>\\n\" }}\n{%- else %}\n    {%- if messages[0]['role'] == 'system' %}\n        {{- '<|im_start|>system\\n' + messages[0]['content'] + '<|im_end|>\\n' }}\n    {%- else %}\n        {{- '<|im_start|>system\\nPlease reason step by step, and put your final answer within \\\\boxed{}.<|im_end|>\\n' }}\n    {%- endif %}\n{%- endif %}\n{%- for message in messages %}\n    {%- if (message.role == \"user\") or (message.role == \"system\" and not loop.first) or (message.role == \"assistant\" and not message.tool_calls) %}\n        {{- '<|im_start|>' + message.role + '\\n' + message.content + '<|im_end|>' + '\\n' }}\n    {%- elif message.role == \"assistant\" %}\n        {{- '<|im_start|>' + message.role }}\n        {%- if message.content %}\n            {{- '\\n' + message.content }}\n        {%- endif %}\n        {%- for tool_call in message.tool_calls %}\n            {%- if tool_call.function is defined %}\n                {%- set tool_call = tool_call.function %}\n            {%- endif %}\n            {{- '\\n<tool_call>\\n{\"name\": \"' }}\n            {{- tool_call.name }}\n            {{- '\", \"arguments\": ' }}\n            {{- tool_call.arguments | tojson }}\n            {{- '}\\n</tool_call>' }}\n        {%- endfor %}\n        {{- '<|im_end|>\\n' }}\n    {%- elif message.role == \"tool\" %}\n        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != \"tool\") %}\n            {{- '<|im_start|>user' }}\n        {%- endif %}\n        {{- '\\n<tool_response>\\n' }}\n        {{- message.content }}\n        {{- '\\n</tool_response>' }}\n        {%- if loop.last or (messages[loop.index0 + 1].role != \"tool\") %}\n            {{- '<|im_end|>\\n' }}\n        {%- endif %}\n    {%- endif %}\n{%- endfor %}\n{%- if add_generation_prompt %}\n    {{- '<|im_start|>assistant\\n' }}\n{%- endif %}\n"
if tokenizer.chat_template is None:
    tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

## Make conversations

Different from DPO, where we need to first apply chat template to form the combination of (instruction, completion), for `GRPOTrainer`, this will be handled within it because the completion is generated online. What we only need to do is to map the raw question into a conversation format (recall that a conversation format contains fields of "role" and "content").

Note that the dataset has already been formatted into the conversation format. As we don't provide our specific system prompt, we can skip the following code cell. In this way, we will use the default system prompt from the

In [ ]:
# Format into conversation
def make_conversation(example, prompt_column: str = "prompt", system_prompt: str = None):
    prompt = []

    if system_prompt is not None:
        prompt.append({"role": "system", "content": system_prompt})

    if prompt_column not in example:
        raise ValueError(f"Dataset Question Field Error: {prompt_column} is not supported.")

    if type(example[prompt_column]) is not list:
        prompt.append({"role": "user", "content": example[prompt_column]})
    else:
        prompt.append(example[prompt_column])

    return {"prompt": prompt}

dataset = raw_datasets.map(make_conversation)

for split in dataset:
    if "messages" in dataset[split].column_names:
        dataset[split] = dataset[split].remove_columns("messages")

In [ ]:
dataset["train"][0]

{'prompt': [[{'content': 'Find the distance from the origin to the curve defined by the equations \\( \\frac{x^2}{4} + y^2 + \\frac{z^2}{4} = 1 \\) and \\( x + y + z = 1 \\) in \\( \\mathbb{R}^3 \\).',
    'role': 'user'}]],
 'solution': '$1$'}

In [ ]:
dataset["test"][0]

{'prompt': [[{'content': 'Evaluate the integral \\( \\int_S y \\, ds \\) where \\( S \\) is the region in the plane \\( z = 1 + y \\) that lies inside the cone \\( z = \\sqrt{2(x^2 + y^2)} \\). Determine the bounds of integration and compute the integral.',
    'role': 'user'}]],
 'solution': '$2\\pi$'}

From the [GRPOTrainer](https://huggingface.co/docs/trl/main/en/grpo_trainer), it expects the data contains "prompt" and "solution" fields to compute the reward. Rename the column of your datasets if you try other datasets.

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 100
    })
    test: Dataset({
        features: ['prompt', 'solution'],
        num_rows: 100
    })
})

## Load SFT model

Here we load the supervised fine-tuned (SFT) model (trained during [step 2](https://github.com/NielsRogge/Transformers-Tutorials/blob/master/Mistral/Supervised_fine_tuning_(SFT)_of_an_LLM_using_Hugging_Face_tooling.ipynb)). As the model size is relatively small, we don't use LoRA this time. But you can definitely train to use LoRA when you want to train a larger model. Please refer to the following doc for details: [lora_without_regret](https://huggingface.co/docs/trl/main/en/lora_without_regret).


In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

device_map = {"": torch.cuda.current_device()} if torch.cuda.is_available() else None

# Load the base model (Qwen2 0.5B Instruct in our case)
model_kwargs = dict(
    # attn_implementation="flash_attention_2", # set this to True if your GPU supports it (Flash Attention drastically speeds up model computations)
    dtype="auto",
    use_cache=False,  # set to False as we're going to use gradient checkpointing
    device_map=device_map
)
base_model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

In [ ]:
base_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## Define Reward Functions

The core to successfully train GRPO is to define proper reward functions. Here we provide two examples: accuracy reward and also format reward.

More rewards functions can be found here: https://github.com/huggingface/open-r1/blob/main/src/open_r1/rewards.py and you can also write your own reward function!

In [ ]:
import asyncio
import json
import math
import re
from functools import partial, update_wrapper
from typing import Callable, Dict, Literal, Optional

from latex2sympy2_extended import NormalizationConfig
from math_verify import LatexExtractionConfig, parse, verify

def accuracy_reward(completions: list[list[dict[str, str]]], solution: list[str], **kwargs) -> list[Optional[float]]:
    """Reward function that checks if the completion is the same as the ground truth."""
    contents = [completion[0]["content"] for completion in completions]
    rewards = []
    for content, sol in zip(contents, solution):
        gold_parsed = parse(
            sol,
            extraction_mode="first_match",
        )
        if len(gold_parsed) != 0:
            # We require the answer to be provided in correct latex (no malformed operators)
            # Set the equations to False to enforce the model's response produces a fully valid LaTeX answer.
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        normalization_config=NormalizationConfig(
                            nits=False,
                            malformed_operators=False,
                            basic_latex=True,
                            equations=False,
                            boxed="all",
                            units=True,
                        ),
                        # Ensures that boxed is tried first
                        boxed_match_priority=0,
                        try_extract_without_anchor=False,
                    )
                ],
                extraction_mode="first_match",
            )
            # Compute binary rewards if verifiable, `None` otherwise to skip this example
            try:
                reward = float(verify(gold_parsed, answer_parsed))
            except Exception as e:
                print(f"verify failed: {e}, answer: {answer_parsed}, gold: {gold_parsed}")
                reward = None
        else:
            # If the gold solution is not parseable, we assign `None` to skip this example
            reward = None
            print("Failed to parse gold solution: ", sol)
        rewards.append(reward)

    return rewards


def format_reward(completions, **kwargs):
    """Reward function that checks if the reasoning process is enclosed within <think> and </think> tags, while the final answer is enclosed within <answer> and </answer> tags."""
    pattern = r"^<think>\n.*?\n</think>\n<answer>\n.*?\n</answer>$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content, re.DOTALL | re.MULTILINE) for content in completion_contents]
    return [1.0 if match else 0.0 for match in matches]

## Define GRPOTrainer

Next, we define the training arguments and instantiate a [GRPOTrainer](https://huggingface.co/docs/trl/main/en/grpo_trainer) class which will handle fine-tuning for us.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

output_dir = "data/qwen2-0.5b-grpo-deepmath"

training_args = GRPOConfig(
    # === regular TrainingArguments-style stuff ===
    output_dir=output_dir,    # It is handy to append `hub_model_revision` to keep track of your local experiments
    bf16=True,
    do_eval=True,
    eval_strategy="steps",
    eval_steps=100,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=5.0e-6,
    log_level="info",
    logging_steps=10,
    lr_scheduler_type="cosine",
    num_train_epochs=20,
    optim="paged_adamw_32bit",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=1,
    seed=42,
    warmup_ratio=0.1,
    # === GRPO related parameters ===
    loss_type="grpo",          # Loss type for GRPO, by default we use the standard GRPO loss. Other alternatives including Dr.GRPO, DAPO, etc.
    max_prompt_length=512,
    max_completion_length=2048,
    num_generations=8
)

trainer = GRPOTrainer(
    model=base_model,
    args=training_args,
    reward_funcs=[accuracy_reward],
    train_dataset=raw_datasets["train"],
    eval_dataset=raw_datasets["test"],
    processing_class=tokenizer
)


The model is already on multiple devices. Skipping the move to device specified in `args`.
Using auto half precision backend


## Train!

Finally, training is as simple as calling trainer.train()!

In [ ]:
train_result = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
***** Running training *****
  Num examples = 100
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 8
  Total optimization steps = 36
  Number of trainable parameters = 494,032,768
Automatic Weights & Biases logging enabled, to disable set os.environ["WANDB_DISABLED"] = "true"
wandb: Currently logged in as: stillarrow to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.27 GiB. GPU 0 has a total capacity of 39.56 GiB of which 8.10 GiB is free. Process 749189 has 30.66 GiB memory in use. Of the allocated memory 20.35 GiB is allocated by PyTorch, and 9.80 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Saving the model

Next, we save the Trainer's state. We also add the number of training samples to the logs.

In [ ]:
metrics = train_result.metrics
max_train_samples = getattr(training_args, "max_train_samples", None)
if max_train_samples is None:
    max_train_samples = len(raw_datasets["train"])
metrics["train_samples"] = min(max_train_samples, len(raw_datasets["train"]))
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
# Save the PEFT adapter
trainer.model.save_pretrained(output_dir)           # or trainer.save_model(output_dir)
# Save tokenizer *from the base model*, not from the adapter
tokenizer.save_pretrained(output_dir)

## Inference

Let's generate some new texts with our trained model.

For inference, there are 2 main ways:
* using the [pipeline API](https://huggingface.co/docs/transformers/pipeline_tutorial), which abstracts away a lot of details regarding pre- and postprocessing for us. [This model card](https://huggingface.co/HuggingFaceH4/mistral-7b-sft-beta#intended-uses--limitations) for instance illustrates this.
* using the `AutoTokenizer` and `AutoModelForCausalLM` classes ourselves and implementing the details ourselves.

Let us do the latter, so that we understand what's going on.

We start by loading the model from the directory where we saved the weights. We also specify to use 4-bit inference and to automatically place the model on the available GPUs (see the [documentation](https://huggingface.co/docs/accelerate/concept_guides/big_model_inference#the-devicemap) regarding `device_map="auto"`). The AutoModelForCausalLM class will automatically load the base model and DPO adapter thanks to the [PEFT integration](https://huggingface.co/docs/peft/tutorial/peft_integrations#transformers) in the Transformers library.

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig
from peft import AutoPeftModelForCausalLM

output_dir = "data/qwen2-0.5b-grpo-deepmath"

# tokenizer is now in output_dir because we saved it there
tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Loads that base model, and then applies the adapter.
model = AutoPeftModelForCausalLM.from_pretrained(
    output_dir,
    device_map="auto",
)
model.eval()

Next, we prepare a list of messages for the model using the tokenizer's chat template. Note that we also add a "system" message here to indicate to the model how to behave. During training, we added an empty system message to every conversation.

We also specify `add_generation_prompt=True` to make sure the model is prompted to generate a response (this is useful at inference time). We specify "cuda" to move the inputs to the GPU. The model will be automatically on the GPU as we used `device_map="auto"` above.

Next, we use the [generate()](https://huggingface.co/docs/transformers/v4.36.1/en/main_classes/text_generation#transformers.GenerationMixin.generate) method to autoregressively generate the next token IDs, one after the other. Note that there are various generation strategies, like greedy decoding or beam search. Refer to [this blog post](https://huggingface.co/blog/how-to-generate) for all details. Here we use sampling.

Finally, we use the batch_decode method of the tokenizer to turn the generated token IDs back into strings.

In [ ]:
import torch

# Set <pad> token if it's not provided to the tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
tokenizer.padding_side = "right"

messages = [
    {"role": "user", "content": "James writes a 3-page letter to 2 different friends twice a week. How many pages does he write a year?"},
]

# Prepare prompt
input_ids = tokenizer.apply_chat_template(
    messages,
    truncation=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

attention_mask = torch.ones_like(input_ids)   # An all-one vector is used as the attention mask because we only have one prompt

# Generate
outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.6,
    top_k=50,
    top_p=0.95,
)

# Only decode the assistant reply
prompt_len = input_ids.shape[1]
generated = outputs[0, prompt_len:]
print(tokenizer.decode(generated, skip_special_tokens=True))
